In [ ]:
import yaml
import pandas as pd

# Load the YAML file
with open('/scicore/home/meiera/schulz0022/projects/growth-and-temperature/orchestration/configs/analysis.yaml', 'r') as file:
    data = yaml.safe_load(file)

# Extract defaults and specifications
defaults = data['analyses']['duckreg']['defaults']
specs = data['analyses']['duckreg']['specifications']

# Create a list of dictionaries for the DataFrame
rows = []
for name, spec in specs.items():
    row = {'name': name}
    # Add defaults
    row.update(defaults)
    # Add spec details
    row.update(spec)
    # Parse the formula if present
    if 'formula' in spec:
        parts = spec['formula'].split(' | ')
        if len(parts) >= 1:
            y_x = parts[0].split(' ~ ')
            if len(y_x) == 2:
                row['dependent'] = y_x[0].strip()
                row['independent'] = y_x[1].strip()
        if len(parts) >= 2:
            row['fixed_effects'] = parts[1].strip()
        if len(parts) >= 3:
            row['instruments'] = parts[2].strip()
        if len(parts) >= 4:
            row['clustering'] = parts[3].strip()
    rows.append(row)

# Create DataFrame
df = pd.DataFrame(rows)

# Drop the original 'formula' column
if 'formula' in df.columns:
    df = df.drop(columns=['formula'])

# Reorder columns: data, formula (parsed parts), query, subset, hyperparams
column_order = ['data_source', 'dependent', 'independent', 'fixed_effects', 'instruments', 'clustering', 'query', 'subset'] + list(defaults.keys())
# Filter to existing columns
column_order = [col for col in column_order if col in df.columns]
df = df[column_order]

# Display the table
df